# Silver: Semantic Review Queue

Manages the queue of jobs requiring manual semantic review.

## Queue Reasons

1. **SECTOR_LOW_CONFIDENCE**: Sector assignment below threshold
2. **TITLE_UNCLEAR**: Job title doesn't match known patterns
3. **SKILL_AMBIGUOUS**: Skills extracted with low confidence
4. **COMPANY_UNKNOWN**: Company not in canonical list
5. **LOCATION_INVALID**: Location parsing failed
6. **DUPLICATE_SUSPECT**: Potential duplicate needing human review

## Workflow

- **PENDING**: Awaiting review
- **IN_REVIEW**: Currently being reviewed
- **RESOLVED**: Issue resolved, notes captured
- **DISMISSED**: Not an actual issue

In [0]:
dbutils.widgets.dropdown("action", "list_pending", ["list_pending", "resolve", "dismiss", "summary"], "Action")
dbutils.widgets.text("review_id", "", "Review ID for resolve/dismiss")
dbutils.widgets.text("resolution_notes", "", "Resolution notes")
dbutils.widgets.text("issue_type_filter", "", "Filter by issue type")

action = dbutils.widgets.get("action")
review_id = dbutils.widgets.get("review_id").strip()
resolution_notes = dbutils.widgets.get("resolution_notes").strip()
issue_type_filter = dbutils.widgets.get("issue_type_filter").strip()

print(f"Action: {action}")
print(f"Review ID: {review_id if review_id else 'N/A'}")
print(f"Issue Type Filter: {issue_type_filter if issue_type_filter else 'All'}")

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {run_id}")

In [0]:
if action == "list_pending":
    query = f"""
    SELECT 
        rq.review_id,
        rq.enterprise_job_id,
        rq.issue_type,
        rq.issue_detail,
        rq.confidence,
        rq.queued_at,
        rq.status,
        j.company_name_raw,
        j.title_raw,
        j.source_name
    FROM {SILVER_SCHEMA}.silver_semantic_review_queue rq
    LEFT JOIN {SILVER_SCHEMA}.silver_jobs_current j
        ON rq.enterprise_job_id = j.enterprise_job_id
    WHERE rq.status = 'PENDING'
    """
    
    if issue_type_filter:
        query += f" AND rq.issue_type = '{issue_type_filter}'"
    
    query += " ORDER BY rq.queued_at ASC LIMIT 100"
    
    pending_df = spark.sql(query)
    pending_count = pending_df.count()
    
    print(f"\n=== Pending Reviews: {pending_count} ===")
    
    if pending_count > 0:
        display(pending_df)
    else:
        print("No pending reviews")
    
    dbutils.notebook.exit({"status": "success", "pending_count": pending_count})

In [0]:
if action == "resolve":
    if not review_id:
        dbutils.notebook.exit({"status": "error", "message": "review_id required for resolve action"})
    
    # Update review queue
    delta_queue = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_semantic_review_queue")
    
    delta_queue.update(
        condition=F.expr(f"review_id = '{review_id}'"),
        set={
            "status": F.lit("RESOLVED"),
            "resolved_at": run_timestamp,
            "resolution_notes": F.lit(resolution_notes if resolution_notes else "Resolved")
        }
    )
    
    print(f"Resolved review item: {review_id}")
    
    result = {
        "status": "success",
        "action": "resolve",
        "review_id": review_id,
        "resolution_notes": resolution_notes
    }
    
    dbutils.notebook.exit(json.dumps(result))

In [0]:
if action == "dismiss":
    if not review_id:
        dbutils.notebook.exit({"status": "error", "message": "review_id required for dismiss action"})
    
    # Update review queue
    delta_queue = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_semantic_review_queue")
    
    delta_queue.update(
        condition=F.expr(f"review_id = '{review_id}'"),
        set={
            "status": F.lit("DISMISSED"),
            "resolved_at": run_timestamp,
            "resolution_notes": F.lit(resolution_notes if resolution_notes else "Not an issue")
        }
    )
    
    print(f"Dismissed review item: {review_id}")
    
    result = {
        "status": "success",
        "action": "dismiss",
        "review_id": review_id
    }
    
    dbutils.notebook.exit(json.dumps(result))

In [0]:
if action == "summary":
    # Overall status breakdown
    status_summary = spark.sql(f"""
    SELECT 
        status,
        COUNT(*) as count,
        MIN(queued_at) as oldest,
        MAX(queued_at) as newest
    FROM {SILVER_SCHEMA}.silver_semantic_review_queue
    GROUP BY status
    ORDER BY count DESC
    """)
    
    print("\n=== Review Queue Status ===")
    display(status_summary)
    
    # Issue type breakdown
    issue_summary = spark.sql(f"""
    SELECT 
        issue_type,
        COUNT(*) as total_count,
        SUM(CASE WHEN status = 'PENDING' THEN 1 ELSE 0 END) as pending_count,
        SUM(CASE WHEN status = 'RESOLVED' THEN 1 ELSE 0 END) as resolved_count,
        AVG(confidence) as avg_confidence
    FROM {SILVER_SCHEMA}.silver_semantic_review_queue
    GROUP BY issue_type
    ORDER BY total_count DESC
    """)
    
    print("\n=== Issue Type Breakdown ===")
    display(issue_summary)
    
    
    result = {"status": "success", "action": "summary"}
    dbutils.notebook.exit(json.dumps(result))